<!--
Copyright (c) 2026 OceanBase.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
-->

# 01 · 让新会话用上项目约定

你在开发订单 CSV 导入器。团队已经决定金额使用整数分，但新会话不知道这件事。我们先保存这一条决定，马上为下一次提问准备上下文，再观察换连接和重启服务之后还剩下什么。

**完成后你能做到：** 保存日常 Memory，为提问准备上下文，用统一制品 API 读取它，并验证跨连接和 Server 重启后的持久性。

预计 10 分钟。先按 [README](README.md) 安装环境；本篇可以独立运行，不依赖其他 Notebook 的变量或数据。本篇无需模型和 API Key。

按顺序读说明、运行代码，再对照结果。练习可以改输入；完整重跑使用 **Restart Kernel & Run All**。

## 准备本篇实验

这格启动一个回环地址的真实 Server，并创建独立 Scope，默认把数据保存在本篇自己的 SQLite 文件中，也可按 [README](README.md#使用-oceanbase-运行) 显式选择专用 OceanBase 测试库。
`_tutorial.py` 只管理环境和显示结果；下面的业务调用都是可在应用中复用的公开 API。

In [ ]:
import sys
from pathlib import Path

from _tutorial import Tutorial, show, table

from powercontext.http import CreateScopeRequest

if not Path("_tutorial.py").is_file():
    sys.path.insert(0, str(Path.cwd() / "examples" / "jupyter"))


if previous_lab := globals().get("lab"):
    await previous_lab.close()
lab = await Tutorial.start("01", features=())
client = lab.client
assert client is not None

scope = await client.create_scope(
    CreateScopeRequest(
        title="订单 CSV 导入器 · 01",
        summary="第 01 篇教程的独立合成数据",
        idempotency_key=f"{lab.run_id}:lesson-01",
    )
)
scope_id = scope.scope_id
show({"title": scope.title, "scope_id": scope_id})

## 1. 没有约定时，先接受空结果

这个项目刚创建，搜索和上下文准备都应是空的。先同时看 `search_memory` 与 `prepare_context`，
确认“没有可召回的知识”和“没有可注入的上下文”是两件可以分别检查的事。
我们明确使用 FTS 主题词；第 09 篇再观察换一种说法之后的语义检索。

In [ ]:
from powercontext.http import PrepareContextRequest, SearchMemoryRequest

empty_search = await client.search_memory(SearchMemoryRequest(scope_id=scope_id, query="amount", mode="fts"))
empty_context = await client.prepare_context(PrepareContextRequest(scope_id=scope_id, query="amount", max_bytes=2500))
assert empty_search.hits == []
assert empty_context.status == "empty" and empty_context.content is None and empty_context.content_bytes == 0
show({"搜索命中": len(empty_search.hits), "上下文状态": empty_context.status})

## 2. 保存应用已经确认的决定

`remember_memory` 接收整理好的内容，直接写入本 Scope 的日常 Memory。它不需要模型，也不要求先提供 Source。
这里由我们明确决定哪些内容值得留下，后续应用可以把这一步放在用户确认项目约定之后。

当前日常召回接口使用 Scope 的默认 Memory。通用 `create_artifact(family="memory")` 会创建另一份独立制品，
因此日常记忆闭环从 `remember_memory` 开始；两种身份的对照放在 HTTP 专题。

In [ ]:
from powercontext.http import RememberMemoryRequest

decision = "amount: 订单金额以整数分存储；100 表示 1 元，禁止用二进制浮点数累计金额。"
saved = await client.remember_memory(
    RememberMemoryRequest(
        scope_id=scope_id,
        kind="decision",
        text=decision,
        reason="团队确认的金额存储约定",
    )
)
assert saved.entry is not None
citation = saved.entry.citation
show({"保存内容": saved.entry.text, "精确引用": citation.model_dump(mode="json")})

## 3. 马上为一次提问准备上下文

应用不必每次加载全部历史。`prepare_context` 根据主题和预算，生成供这次模型请求使用的材料。
下面将真正返回的内容放进待发送消息；本篇只检查消息已经准备好，第 10 篇再验证真实模型是否收到并使用它。

In [ ]:
from powercontext.http import PrepareContextRequest

prepared = await client.prepare_context(PrepareContextRequest(scope_id=scope_id, query="amount", max_bytes=2500))
assert prepared.status == "ready" and prepared.content is not None
assert "整数分" in prepared.content
messages = [
    {"role": "system", "content": "帮助开发订单导入器。历史材料需结合当前要求和实际检查使用。"},
    {"role": "system", "content": prepared.content},
    {"role": "user", "content": "amount 字段应该用什么单位存储？"},
]
table([{"角色": item["role"], "内容": item["content"]} for item in messages])

## 4. 换一个连接，读取已经保存的制品

我们用新的 Python Client 模拟应用的下一次会话。它通过 Server 读取数据，不复制上一次返回的 Memory 内容。
新连接与独立 Python 进程仍是不同概念；这里验证的是连接结束后服务数据仍然存在。

`citation.memory_ref` 给出这份 Memory 的制品身份。使用统一的 `get_artifact`，就能读取当前版。
同一套读取接口也适用于后面会创建的 Experience、Skill 和 Handoff。

In [ ]:
from powercontext.client import PowerContextClient

async with PowerContextClient(lab.base_url) as next_session:
    found = await next_session.search_memory(SearchMemoryRequest(scope_id=scope_id, query="amount", mode="fts"))
    artifact = await next_session.get_artifact(scope_id, "memory", citation.memory_ref.artifact_id)
assert any(hit.text == decision for hit in found.hits)
assert artifact is not None and artifact.revision == citation.memory_ref.revision
show({"新连接找回约定": True, "制品类型": artifact.family, "制品 ID": artifact.artifact_id, "版本": artifact.revision})

## 5. 服务重启之后，旧引用还有效吗？

`lab.restart()` 真正停止并重新启动 Server，复用同一个存储后端。随后读取刚才保存的精确制品版本。
这次请求证明数据可以跨服务生命周期保留，不能仅靠客户端变量解释结果。

In [ ]:
await lab.restart()
client = lab.client
assert client is not None
restored = await client.get_artifact_revision(
    scope_id,
    "memory",
    citation.memory_ref.artifact_id,
    citation.memory_ref.revision,
)
assert artifact is not None and restored.content == artifact.content
show({"Server 重启后仍能读取": True, "精确版本": restored.revision})

## 练习：再保存一种约定

把 `my_constraint` 改成自己的项目约束，保留主题词 `line_number`，再观察检索结果。
这条新知识是否有自己的 `entry_id`？它是否仍属于同一份日常 Memory 制品？

In [ ]:
my_constraint = "line_number: 导入坏行时必须返回原始 CSV 行号，方便用户修复输入。"
practice = await client.remember_memory(
    RememberMemoryRequest(
        scope_id=scope_id,
        kind="constraint",
        text=my_constraint,
        reason="教程练习",
    )
)
check = await client.search_memory(SearchMemoryRequest(scope_id=scope_id, query="line_number", mode="fts"))
assert practice.entry is not None
assert any(hit.citation.entry_id == practice.entry.citation.entry_id for hit in check.hits)
show({"自己的约束已被找回": True})
assert practice.entry.citation.entry_id != citation.entry_id
assert practice.entry.citation.memory_ref.artifact_id == citation.memory_ref.artifact_id
print("两个不同条目属于同一份日常 Memory 制品。")

## 保存收获，关闭连接

已经走通保存 → 准备上下文 → 新连接读取 → 服务重启读取。下一步学习：同一条约定变化后，如何保留历史并处理并发编辑。

下面关闭本篇 Client 和 Server，保留实验文件供检查。中途停止时也可运行这一格；清理方式见 [README](README.md#清理实验数据)。

下一篇：[02](02_memory_lifecycle.ipynb)。

In [ ]:
await lab.close()
print("本篇 Server 已关闭。")